In this notebook there is the code needed for
 * Gaussian Mixture analysis
 * Figure 5.1: cost vs rounds for 10% of KDDcup1999
 * Figure 5.2: cost vs rounds for Gaussian Mixture

## 1. Import e flag

In [19]:
import os, time
import numpy as np
import pandas as pd

from src.launch_cluster import launch_cluster, shutdown_cluster
from src.data_loader import load_dataset, make_gauss_mixture, array_to_bag
from src.paper_experiments import (
    run_fig51, run_fig52, run_table34,
    plot_fig51, plot_fig52, table34_cost_table, table34_time_table,
)
from src.benchmark import RESULTS_DIR

# --- flag di esecuzione: accendere UNA sezione alla volta ---
RUN_FIG52_TINY   = False   # griglia ridotta, locale (~1 min)
RUN_FIG52_FULL   = False   # griglia completa dell'articolo, locale (~ore)
RUN_CLUSTER      = True   # abilita le sezioni [CLUSTER]
RUN_FIG51        = False  # KDD 10%, exact-l (cluster, costo moderato)
RUN_TABLE_SANITY = False   # singola run k=500 l/k=10 PRIMA della sweep piena
RUN_TABLE34      = True   # KDD full (cluster, NOTTATA)

SEED = 42
N_RUNS = 11                # convenzione del paper: mediana su 11 run

In [1]:
from dask.distributed import Client

# si collega allo scheduler già attivo (HEAD_IP:SCHEDULER_PORT)
client = Client("10.67.22.194:8786")

client.shutdown()   # ferma scheduler + tutti i worker

## 2. Fig 5.2 — GaussMixture **[LOCALE]**

Griglia ridotta di validazione prima della corsa completa.

In [6]:
if RUN_FIG52_TINY:
    #df52 = run_fig52(client=None,
                  #   R_values=(1,), l_over_k_values=(1.0, 2.0),
                   #  r_values=(0, 1, 2, 3), k=20, n=2_000, d=8,
                    # n_runs=2, seed=SEED, max_iter_fit=30)
    df52 = pd.read_csv("/home/ubuntu/Project/libero_development/results/with_old_code/other_old//fig52_tiny.csv")
    df52.to_csv(os.path.join(RESULTS_DIR, "fig52_tiny.csv"), index=False)
    display(df52.groupby(["method", "l_over_k", "r"])["cost_final"].median())
    plot_fig52(df52, output_dir="figures")
else:
    print("RUN_FIG52_TINY = False")

method    l_over_k  r  
kmeans||  1.0       0.0    14194.672424
                    1.0    14102.990022
                    2.0    14139.888756
                    3.0    14232.421254
          2.0       0.0    14194.672424
                    1.0    14237.784236
                    2.0    14162.106413
                    3.0    14178.602777
Name: cost_final, dtype: float64

Saved figures/fig52_cost_vs_rounds.png


Corsa completa (protocollo articolo: R ∈ {1,10,100}, ℓ/k ∈ {0.1,…,10},
r = 0..15, mediana su 11 run, k=50). Attenzione: ore di CPU locali.

In [3]:
if RUN_FIG52_FULL:
    df52 = run_fig52(client=None, seed=SEED, n_runs=N_RUNS)
    _ts = time.strftime("%Y%m%d_%H%M%S")
    _csv = os.path.join(RESULTS_DIR, f"fig52_{_ts}.csv")
    df52.to_csv(_csv, index=False)
    print("Saved", _csv)
    plot_fig52(df52, output_dir="figures")
else:
    print("RUN_FIG52_FULL = False")

[fig52] R=1: riferimento k-means++, 11 run
 -> SEEDING FAILED (candidati < k, k=50, l=5, r=1): n_samples=4 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=5, r=1): n_samples=3 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=5, r=1): n_samples=7 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=5, r=1): n_samples=6 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=5, r=1): n_samples=5 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=5, r=1): n_samples=2 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=5, r=1): n_samples=5 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=5, r=1): n_samples=8 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=5, r=1): n_samples=6 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=5, r=1): n_samples=12 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k

/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 1 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(


[fig52] R=10, l=5 (l/k=0.1): r sweep completato
 -> SEEDING FAILED (candidati < k, k=50, l=25, r=1): n_samples=25 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=25, r=1): n_samples=24 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=25, r=1): n_samples=19 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=25, r=1): n_samples=27 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=25, r=1): n_samples=23 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=25, r=1): n_samples=19 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=25, r=1): n_samples=22 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=25, r=1): n_samples=27 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=25, r=1): n_samples=26 should be >= n_clusters=50.
 -> SEEDING FAILED (candidati < k, k=50, l=25, r=1): n_samples=36 should be >= n_clusters=50.
 -> SEEDING 

/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 1 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(


[fig52] R=10, l=50 (l/k=1): r sweep completato
[fig52] R=10, l=100 (l/k=2): r sweep completato
[fig52] R=10, l=500 (l/k=10): r sweep completato
[fig52] R=100: riferimento k-means++, 11 run


...

[fig52] R=100, l=50 (l/k=1): r sweep completato


/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 2 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(
/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 1 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(
/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 1 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(
/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 1 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(
/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 1 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iter

[fig52] R=100, l=100 (l/k=2): r sweep completato


/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 2 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(
/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 1 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(


[fig52] R=100, l=500 (l/k=10): r sweep completato
Saved /home/ubuntu/Project/libero_development/results/fig52_20260909_103719.csv
Saved figures/fig52_cost_vs_rounds.png


## 3. Dati KDD **[CLUSTER]** — caricamento (come in analysis.ipynb)

Paths identici alle VM (NON modificare, vedi AGENTS.md). Per il 10% usare
`DATASET_URL_10PC`; per le tabelle usare `DATASET_URL_FULL`.

In [20]:
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
DATASET_URL_FULL = "https://ndownloader.figshare.com/files/5976045"

RAW_GZ_PATH  = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz"
PARQUET_PATH = "/tmp/kddcup_data.parquet"

COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label",
]

### Cluster on/off

In [21]:
# DO NOT RUN if already existing!
if RUN_CLUSTER:
    N_WORKERS = 8
    NUM_PARTITIONS = 8 * N_WORKERS
    cluster, client = launch_cluster(N_WORKERS)
else:
    print("RUN_CLUSTER = False")

Inizializzazione del cluster SSH con 8 worker...
Worker selezionati: ['10.67.22.254', '10.67.22.34', '10.67.22.145', '10.67.22.121', '10.67.22.192', '10.67.22.18', '10.67.22.187', '10.67.22.48']


2026-09-11 08:52:40,486 - distributed.deploy.ssh - INFO - 2026-09-11 08:52:40,485 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-09-11 08:52:40,517 - distributed.deploy.ssh - INFO - 2026-09-11 08:52:40,516 - distributed.scheduler - INFO - State start
2026-09-11 08:52:40,522 - distributed.deploy.ssh - INFO - 2026-09-11 08:52:40,521 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.67.22.194:8786
2026-09-11 08:52:42,250 - distributed.deploy.ssh - INFO - 2026-09-11 08:52:42,252 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.121:42377'
2026-09-11 08:52:42,306 - distributed.deploy.ssh - INFO - 2026-09-11 08:52:42,307 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.145:42675'
2026-09-11 08:52:42,307 - distributed.deploy.ssh - INFO - 2026-09-11 08:52:42,308 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.25

Cluster avviato e connessione stabilita con successo!



## 4. Fig 5.1 — KDD 10%, exact-ℓ **[CLUSTER]**

Protocollo: k ∈ {17,33,65,129}, ℓ/k ∈ {1,2,4}, r = 1..10, mediana su 11 run.
Costo moderato: ~4·3·10·11 = 1320 run di seeding+Lloyd's sul 10%.

In [ ]:
if RUN_CLUSTER and RUN_FIG51:
    #X_bag, _ = load_dataset(DATASET_URL_10PC, RAW_GZ_PATH, PARQUET_PATH,
                            PARQUET_PATH, COL_NAMES,
                            n_partitions=NUM_PARTITIONS, client=client)
    #df51 = run_fig51(client, X_bag, seed=SEED, n_runs=N_RUNS,
                     num_partitions=NUM_PARTITIONS)
    df51.to_csv(os.path.join(RESULTS_DIR, "fig51_full.csv"), index=False)
    plot_fig51(df51, output_dir="figures")
else:
    print("RUN_CLUSTER/RUN_FIG51 = False")

Using cached dataset: /home/ubuntu/Project/libero_development/data/kddcup_data.gz
Converting .gz -> Parquet chunk sizes...
Parquet file created (compressed with snappy).
Number of partitions before preprocessing: 64
Constant columns: ['num_outbound_cmds', 'is_host_login']
Computing global mean and std (first pass over data)...
Distributed bag created with 32 partitions.
Number of samples: 494021


/home/ubuntu/pyvenv/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (13) found smaller than n_clusters (17). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:729: UserWarning: 4 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(


[fig51] k=17, l=17 (l/k=1), r=1: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=2: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=3: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=4: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=5: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=6: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=7: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=8: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=9: 11 run completate


SE VUOI SOLO PLOTTARE !!

In [4]:
if RUN_CLUSTER and RUN_FIG51:
    df51 = pd.read_csv("/home/ubuntu/Project/libero_development/results/with_old_code/other_old/fig51_full.csv")  # <-- il path che hai trovato
    plot_fig51(df51, output_dir="figures")
else:
    print("RUN_CLUSTER/RUN_FIG51 = False")

Saved figures/fig51_cost_vs_rounds.png


## Cost/Time vs ℓ/k — KDD 10%, r=5  **[CLUSTER]**

Same protocol as Table 3/4 (`policy="fixed", r=5`) but on the 10% sample of
KDDCup1999 instead of the full dataset: ℓ/k ∈ {0.1, 0.5, 1, 2}, k ∈ {500, 1000},
mean ± std over N_RUNS repetitions.


In [9]:
from src.paper_experiments import run_l_sweep, plot_l_sweep

PARQUET_PATH_LK = "/tmp/kddcup_data_lk.parquet"
if RUN_CLUSTER:
    X_bag_10pct, _ = load_dataset(DATASET_URL_10PC, RAW_GZ_PATH, PARQUET_PATH_LK,
                                   PARQUET_PATH_LK, COL_NAMES,
                                   n_partitions=NUM_PARTITIONS, client=client)

    df_lk = run_l_sweep(client, X_bag_10pct,
                        k_values=(500, 1000),
                        l_over_k_values=(10,20,100),
                        r_fixed=5, n_runs=N_RUNS, seed=SEED,
                        num_partitions=NUM_PARTITIONS,
                        policy="auto")
    df_lk.to_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct_extra.csv"), index=False)
    #plot_l_sweep(df_lk, output_dir="figures")
else:
    print("RUN_CLUSTER = False")


Using cached dataset: /home/ubuntu/Project/libero_development/data/kddcup_data.gz
Converting .gz -> Parquet chunk sizes...
Parquet file created (compressed with snappy).
Number of partitions before preprocessing: 64
Constant columns: ['num_outbound_cmds', 'is_host_login']
Computing global mean and std (first pass over data)...
Distributed bag created with 32 partitions.
Number of samples: 494021
Stopped at LLoyd's iteration 5 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 11 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 11 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 6 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 7 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 12 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 7 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 7 due to relative tolerance threshold 0.0001
Stoppe

/home/ubuntu/pyvenv/lib/python3.10/site-packages/distributed/client.py:3415: UserWarning: Sending large graph of size 11.32 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Stopped at LLoyd's iteration 0 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 0 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 0 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 0 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 0 due to relative tolerance threshold 0.0001
[l_sweep] k=500, l=50000 (l/k=100), r=5: 11 runs done
Stopped at LLoyd's iteration 4 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 3 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 2 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 4 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 7 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 7 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 8 due to relative tolerance threshold 0.0001
Stopped at LLoyd's iteration 5 due to relative tolerance t

/home/ubuntu/pyvenv/lib/python3.10/site-packages/distributed/client.py:3415: UserWarning: Sending large graph of size 17.14 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Stopped at LLoyd's iteration 0 due to relative tolerance threshold 0.0001


...

Stopped at LLoyd's iteration 0 due to relative tolerance threshold 0.0001


/home/ubuntu/pyvenv/lib/python3.10/site-packages/distributed/client.py:3415: UserWarning: Sending large graph of size 17.05 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Stopped at LLoyd's iteration 0 due to relative tolerance threshold 0.0001


/home/ubuntu/pyvenv/lib/python3.10/site-packages/distributed/client.py:3415: UserWarning: Sending large graph of size 17.12 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Stopped at LLoyd's iteration 0 due to relative tolerance threshold 0.0001
[l_sweep] k=1000, l=100000 (l/k=100), r=5: 11 runs done


In [6]:
df_lk = pd.read_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct.csv"))
df_lk_clean = df_lk[df_lk["l_over_k"] != 0.2]
plot_l_sweep(df_lk_clean, output_dir="figures")

df_lk_clean.to_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct_no02.csv"), index=False)


Saved figures/l_sweep_k500.png
Saved figures/l_sweep_k1000.png


In [10]:
df_base = pd.read_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct_no02.csv"))
df_extra = pd.read_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct_extra.csv"))

df_all = (pd.concat([df_base, df_extra], ignore_index=True)
            .sort_values(["k", "l_over_k", "run"])
            .reset_index(drop=True))

df_all.to_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct_full.csv"), index=False)
print(f"{len(df_base)} + {len(df_extra)} -> {len(df_all)} rows")
print(df_all.groupby(["k", "l_over_k"]).size())

110 + 66 -> 176 rows
k     l_over_k
500   0.1         11
      0.5         11
      1.0         11
      2.0         11
      5.0         11
      10.0        11
      20.0        11
      100.0       11
1000  0.1         11
      0.5         11
      1.0         11
      2.0         11
      5.0         11
      10.0        11
      20.0        11
      100.0       11
dtype: int64


In [17]:
plot_l_sweep_log(df_all, output_dir="figures",logx=True)

Saved figures/l_sweep_k500.png
Saved figures/l_sweep_k1000.png


['figures/l_sweep_k500.png', 'figures/l_sweep_k1000.png']

In [16]:
import matplotlib.pyplot as plt
def plot_l_sweep_log(results, output_dir="figures", dpi=150, logx=False, logy=False):
    """One figure per k: final cost (left) and total running time (right)
    vs l/k, mean +/- std over the runs. ``results`` is a DataFrame or a CSV
    path (as produced by ``run_l_sweep``). ``logx``/``logy`` set log scale on
    the respective axes (useful when l/k spans decades).
    Returns the list of saved paths."""
    df = _load_csv(results) if isinstance(results, str) else results.copy()
    if "artifact" in df.columns:
        df = df[df["artifact"] == "l_sweep"]
    if "failed" in df.columns:
        df = df[df["failed"] == False]  # noqa: E712 (pandas mask)
    if "time_total" not in df.columns:
        df = df.assign(time_total=df["time_seed"] + df["time_fit"])

    metrics = [("cost_final", "Cost"), ("time_total", "Time (s)")]
    colors = {"cost_final": "tab:blue", "time_total": "tab:orange"}
    outpaths = []
    for k in sorted(df["k"].unique()):
        g = (df[df["k"] == k]
             .groupby("l_over_k")[["cost_final", "time_total"]]
             .agg(["mean", "std"]))
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
        for ax, (col, ylab) in zip(axes, metrics):
            ax.errorbar(g.index, g[(col, "mean")], yerr=g[(col, "std")],
                        marker="o", capsize=4, color=colors[col])
            ax.set_xlabel(r"$\ell\,/\,k$")
            ax.set_ylabel(f"{ylab} (mean $\\pm$ std)")
            ax.set_title(f"{ylab.split(' (')[0]} vs $\\ell/k$  ($k={k}$)")
            if logx:
                ax.set_xscale("log")
            if logy:
                ax.set_yscale("log")
            ax.grid(True, alpha=0.3, which="both")
        fig.suptitle(f"Cost and running time vs oversampling factor "
                     f"$\\ell/k$  ($k = {k}$)", fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        outpath = os.path.join(output_dir, f"l_sweep_k{k}.png")
        fig.savefig(outpath, dpi=dpi)
        plt.close(fig)
        print("Saved", outpath)
        outpaths.append(outpath)
    return outpaths


## 5. Table 3/4 — KDD full **[CLUSTER, nottata]**

Prima il sanity check (rischio #3 di ANALYSIS_PLAN: pool candidati grandi con
ℓ=10k), poi la sweep completa. Protocollo critico: `policy="fixed", r=5`
(gestito dentro `run_table34`) — la regola auto ℓ/k≤0.1→15 NON è quella
della tabella.

In [ ]:
if RUN_CLUSTER and RUN_TABLE_SANITY:
    X_bag_full, _ = load_dataset(DATASET_URL_FULL, RAW_GZ_PATH, PARQUET_PATH,
                                 PARQUET_PATH, COL_NAMES,
                                 n_partitions=NUM_PARTITIONS, client=client)
    # singola run piu' pesante: k=500, l=5000 -> pool atteso ~25k candidati
    from src.paper_experiments import _run_one_parallel
    import time as _t
    _t0 = _t.time()
    res = _run_one_parallel(X_bag_full, k=500, l=5000, r=5, run_seed=SEED,
                            policy="fixed", max_iter_fit=10)
    print(res)
    print(f"totale {_t.time()-_t0:.1f}s")
else:
    print("RUN_CLUSTER/RUN_TABLE_SANITY = False")

In [ ]:
if RUN_CLUSTER and RUN_TABLE34:
    df34 = run_table34(client, X_bag_full, seed=SEED, n_runs=N_RUNS,
                       num_partitions=NUM_PARTITIONS)
    df34.to_csv(os.path.join(RESULTS_DIR, "table34_full.csv"), index=False)
else:
    print("RUN_CLUSTER/RUN_TABLE34 = False")

## 6. Analisi: figure e tabelle stile-articolo

Dai CSV salvati (funziona anche su questa macchina dopo aver riportato i CSV
in `results/`, oppure direttamente dai DataFrame delle sezioni sopra).

In [ ]:
# Esempio (decommentare quando i CSV esistono):
# p51 = os.path.join(RESULTS_DIR, "fig51_full.csv")
# p52 = os.path.join(RESULTS_DIR, "fig52_<timestamp>.csv")
# p34 = os.path.join(RESULTS_DIR, "table34_full.csv")
# plot_fig51(p51, output_dir="figures")
# plot_fig52(p52, output_dir="figures")
# display(table34_cost_table(p34))     # Table 3, costi x1e-10 (mediane)
# display(table34_time_table(p34))     # Table 4, tempi (mediane)

## 7. Confronto con i valori del paper

Compilare dopo ogni artefatto: tabella obtained-vs-paper (costi ×10⁻¹⁰,
mediane). Valori di riferimento nel PDF: `docs/1203.6402v1.pdf`,
Tabelle 3/4 e Figures 5.1/5.2.

| Artefatto | Configurazione | Paper | Ottenuto | Note |
|---|---|---|---|---|
| Table 3 | k=500, ℓ/k=1, r=5, final | *(da PDF)* | | |
| ... | | | | |

## 8. Spegnimento cluster

In [7]:
# Da eseguire a fine lavoro
shutdown_cluster(cluster, client)

Cluster e client chiusi.


In [9]:
from dask.distributed import Client
c = Client("10.67.22.194:8786", timeout=10)
c.shutdown()


KeyboardInterrupt: 